### とにかく「shit + Enter」を押していくと、上から実行していきます。


In [ ]:
# 計算できます。「#」以下は無視されます。
1+2

In [ ]:
# 代入できます。最後の行だけ出力が見えます。
a = 2
b = 3
a**b # これは２の３乗という意味です。

In [ ]:
# 便利な関数たちを呼び出して使えます。
import numpy as np # 数値計算用（Numerical Pythonの略）
import matplotlib.pyplot as plt  # グラフプロット用

x = np.arange(0, 3, 0.1) #0から3まで0.1刻み。3は含まれないことに注意。
y = a*(x**2) + b
plt.plot( x, y , 'o-' )
print(x)  # printをつけると最後の行じゃなくても出力します。
print(y)  # printをつけると最後の行じゃなくても出力します。


### いきなりですが、データを統計解析してみます


In [ ]:
# ① CSVを読み込む
import pandas as pd
from scipy import stats

url = 'https://raw.githubusercontent.com/hosodakazufumi/note/main/data/phq9_training_160.csv'
df = pd.read_csv(url)

df


In [ ]:
# 列を足す
df["pre_ge10"] = (df["pre_total"] >= 10).astype(int)
df["week8_ge10"] = (df["week8_total"] >= 10).astype(int)


In [ ]:
# ② データを眺める
df.head() # 上だけ

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# とりあえずプロット
import seaborn as sns

sns.boxenplot(data=df, x="group", y="improvement")


In [ ]:
# ③ 記述統計
df[["age", "pre_total", "week8_total", "improvement"]].describe()

In [ ]:
# ④ 4群の平均・中央値を比較
df.groupby("group")["improvement"].agg( ["count", "mean", "median", "std"] )


In [ ]:
# ⑤ 同じ人の介入前後を比較

# A群の改善量
a = df[df["group"] == "A"]
diff = a["pre_total"] - a["week8_total"]

# 対応のあるウィルコクソン検定
stats.wilcoxon( diff )


In [ ]:
# 符号検定
stats.binomtest( (diff > 0).sum(), (diff != 0).sum(), p=0.5)


In [ ]:
# ⑥ 4群の差を検定

groups = ["A", "B", "C", "D"]

samples = [ df[df["group"] == g]["improvement"]  for g in groups ]

stats.kruskal(*samples)


In [ ]:
# ⑦ 多重比較：Manwihtneyu + Holm法

from itertools import combinations
from statsmodels.stats.multitest import multipletests

pairs = list(combinations(groups, 2))

p_values = []
for g1, g2 in pairs:
    x = df[df["group"] == g1]["improvement"]
    y = df[df["group"] == g2]["improvement"]

    p = stats.mannwhitneyu(x, y).pvalue
    p_values.append(p)

# Holm法による多重比較補正
adjusted = multipletests(p_values, method="holm")[1]

pd.DataFrame({
    "比較": [f"{a}-{b}" for a, b in pairs],
    "補正前p": p_values,
    "Holm補正後p": adjusted,
    "Bonferroni補正後p": np.array(p_values)*len(p_values) # (実際にはαを割るのですが、今回は同じことなのでpにかけた)
})



In [ ]:
# 初回のみ
!pip install scikit-posthocs

In [ ]:
# Dunn + Holm法

import scikit_posthocs as sp

data = df[
    df["group"].isin(["A", "B", "C", "D"])
][["group", "improvement"]].dropna()

result = sp.posthoc_dunn(
    data,
    val_col="improvement",
    group_col="group",
    p_adjust="holm"
)

print(result)

In [ ]:
# Steel-Dwass系の多重比較（DSCF）
import scikit_posthocs as sp

data = df[df["group"].isin(["A", "B", "C", "D"])]
data = data[["group", "improvement"]].dropna()

result = sp.posthoc_dscf(
    data,
    val_col="improvement",
    group_col="group"
)

print(result)

In [ ]:
# カイ二乗検定・フィッシャー検定

# 4群の割合の違い
table = pd.crosstab(df["group"], df["week8_ge10"])

print(stats.chi2_contingency(table))

# A群とD群だけの割合を比較
ad = df[df["group"].isin(["A", "D"])]
table_ad = pd.crosstab(ad["group"], ad["week8_ge10"])

print(stats.fisher_exact(table_ad))


In [ ]:
# マクネマー検定

from statsmodels.stats.contingency_tables import mcnemar

# D群の同じ患者の前後変化
d = df[df["group"] == "D"]

table = pd.crosstab(
    d["pre_ge10"],
    d["week8_ge10"]
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

print(table)
print(mcnemar(table, exact=True))


In [ ]:
# スピアマンの順位相関

rho, p = stats.spearmanr(
    df["pre_total"],
    df["week8_total"]
)

print("順位相関係数:", rho)
print("p値:", p)


In [ ]:
# 線形回帰・ロジスティック回帰

import statsmodels.formula.api as smf

# 線形回帰：8週間後のPHQ-9合計点を説明
model1 = smf.ols(
    "week8_total ~ pre_total + age + C(group)",
    data=df
).fit()

print(model1.summary())

# ロジスティック回帰：8週間後に10点以上かを説明
model2 = smf.logit(
    "week8_ge10 ~ pre_total + age + C(group)",
    data=df
).fit(disp=False)

print(model2.summary())
